# OMERO Functions

In [2]:
import omero
from omero.gateway import BlitzGateway
from omero.model import DatasetI, ProjectI, ImageI
from omero.rtypes import rstring, rlong, rint
import numpy as np
from PIL import Image
import os
from typing import Optional, Dict, List, Tuple
from pathlib import Path
from dotenv import load_dotenv

2025-10-20 17:07:23,337 DEBUG [              omero.util.TempFileManager] (MainThread) Added file /Users/austinwu/omero/tmp/.lock_testpq0iaegn.tmp
2025-10-20 17:07:23,338 DEBUG [              omero.util.TempFileManager] (MainThread) Chose global tmpdir: /Users/austinwu/omero/tmp
2025-10-20 17:07:23,338 DEBUG [              omero.util.TempFileManager] (MainThread) Using temp dir: /Users/austinwu/omero/tmp/omero_austinwu/74337


## Connection

In [3]:
def connect_to_omero(host: str, username: str, password: str, port: int = 4064) -> BlitzGateway:
    conn = BlitzGateway(username, password, host=host, port=port, secure=True)
    if not conn.connect():
        raise ConnectionError("Failed to connect to OMERO server")
    return conn

def disconnect_from_omero(conn: BlitzGateway):
    if conn:
        conn.close()

## Upload

In [ ]:
def upload_image(conn: BlitzGateway, image_data, dataset_id: Optional[int] = None, image_name: str = "image") -> int:
    """Upload image from file path or numpy array"""
    if isinstance(image_data, (str, Path)):
        import subprocess
        session_key = conn.getSession().getUuid().val
        cmd = ["omero", "-s", conn.host, "-k", session_key, "import"]
        if dataset_id:
            cmd.extend(["-d", str(dataset_id)])
        cmd.append(str(image_data))
        result = subprocess.run(cmd, capture_output=True, text=True)
        if result.returncode != 0:
            raise RuntimeError(f"Upload failed: {result.stderr}")
    else:
        if isinstance(image_data, np.ndarray):
            img_array = image_data
        else:
            img_array = np.array(image_data)
        
        if img_array.ndim == 2:
            size_z, size_c, size_t = 1, 1, 1
            size_y, size_x = img_array.shape
            img_array = img_array.reshape(1, 1, 1, size_y, size_x)
        elif img_array.ndim == 3:
            size_c, size_y, size_x = img_array.shape
            size_z, size_t = 1, 1
            img_array = img_array.reshape(1, size_c, 1, size_y, size_x)
        elif img_array.ndim == 4:
            size_z, size_c, size_y, size_x = img_array.shape
            size_t = 1
            img_array = img_array.reshape(size_z, size_c, 1, size_y, size_x)
        else:
            size_z, size_c, size_t, size_y, size_x = img_array.shape
        
        def plane_gen():
            for t in range(size_t):
                for c in range(size_c):
                    for z in range(size_z):
                        yield img_array[z, c, t, :, :].astype(img_array.dtype)
        
        image = conn.createImageFromNumpySeq(
            plane_gen(), image_name, size_z, size_c, size_t,
            description=None, dataset=None
        )
        image_id = image.getId()
        
        if dataset_id:
            dataset = conn.getObject("Dataset", dataset_id)
            link = omero.model.DatasetImageLinkI()
            link.setParent(omero.model.DatasetI(dataset_id, False))
            link.setChild(omero.model.ImageI(image_id, False))
            conn.getUpdateService().saveObject(link)
        
        return image_id
    
    if dataset_id:
        dataset = conn.getObject("Dataset", dataset_id)
        images = list(dataset.listChildren())
        if images:
            return images[-1].getId()
    return -1

In [5]:
def create_dataset(conn: BlitzGateway, name: str, description: Optional[str] = None, project_id: Optional[int] = None) -> int:
    dataset = omero.model.DatasetI()
    dataset.setName(rstring(name))
    if description:
        dataset.setDescription(rstring(description))
    dataset = conn.getUpdateService().saveAndReturnObject(dataset)
    dataset_id = dataset.getId().getValue()
    
    if project_id:
        link = omero.model.ProjectDatasetLinkI()
        link.setParent(omero.model.ProjectI(project_id, False))
        link.setChild(omero.model.DatasetI(dataset_id, False))
        conn.getUpdateService().saveObject(link)
    
    return dataset_id

def create_project(conn: BlitzGateway, name: str, description: Optional[str] = None) -> int:
    project = omero.model.ProjectI()
    project.setName(rstring(name))
    if description:
        project.setDescription(rstring(description))
    project = conn.getUpdateService().saveAndReturnObject(project)
    return project.getId().getValue()

## Download

In [6]:
def download_image(conn: BlitzGateway, image_id: int, output_path: str) -> str:
    image = conn.getObject("Image", image_id)
    if not image:
        raise ValueError(f"Image {image_id} not found")
    
    output_path = Path(output_path)
    output_path.parent.mkdir(parents=True, exist_ok=True)
    
    fileset = image.getFileset()
    if fileset:
        for orig_file in fileset.listFiles():
            raw_file_store = conn.createRawFileStore()
            try:
                raw_file_store.setFileId(orig_file.getId().getValue())
                with open(output_path, 'wb') as f:
                    offset = 0
                    size = orig_file.getSize().getValue()
                    chunk_size = 1024 * 1024
                    while offset < size:
                        chunk = raw_file_store.read(offset, min(chunk_size, size - offset))
                        f.write(chunk)
                        offset += len(chunk)
                return str(output_path)
            finally:
                raw_file_store.close()
    
    pixels = image.getPrimaryPixels()
    planes = [pixels.getPlane(z, c, t) 
              for z in range(image.getSizeZ()) 
              for c in range(image.getSizeC()) 
              for t in range(image.getSizeT())]
    
    img = Image.fromarray(planes[0])
    if len(planes) > 1:
        img.save(output_path, save_all=True, append_images=[Image.fromarray(p) for p in planes[1:]])
    else:
        img.save(output_path)
    return str(output_path)

def download_dataset(conn: BlitzGateway, dataset_id: int, output_dir: str) -> List[str]:
    dataset = conn.getObject("Dataset", dataset_id)
    if not dataset:
        raise ValueError(f"Dataset {dataset_id} not found")
    
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)
    
    downloaded_files = []
    for image in dataset.listChildren():
        safe_name = "".join(c for c in image.getName() if c.isalnum() or c in (' ', '.', '_', '-'))
        output_path = output_dir / f"{safe_name}_{image.getId()}.tiff"
        try:
            downloaded_files.append(download_image(conn, image.getId(), str(output_path)))
        except Exception as e:
            print(f"Error downloading image {image.getId()}: {e}")
    
    return downloaded_files

## Metadata

In [7]:
def add_key_value_pairs(conn: BlitzGateway, object_type: str, object_id: int, key_value_data: Dict[str, str]) -> int:
    map_ann = omero.gateway.MapAnnotationWrapper(conn)
    map_ann.setNs(omero.constants.metadata.NSCLIENTMAPANNOTATION)
    map_ann.setValue([[str(k), str(v)] for k, v in key_value_data.items()])
    map_ann.save()
    obj = conn.getObject(object_type, object_id)
    if not obj:
        raise ValueError(f"{object_type} {object_id} not found")
    obj.linkAnnotation(map_ann)
    return map_ann.getId()

def get_key_value_pairs(conn: BlitzGateway, object_type: str, object_id: int) -> Dict[str, str]:
    obj = conn.getObject(object_type, object_id)
    if not obj:
        raise ValueError(f"{object_type} {object_id} not found")
    all_kv = {}
    for ann in obj.listAnnotations():
        if isinstance(ann, omero.gateway.MapAnnotationWrapper):
            for key, value in ann.getValue():
                all_kv[key] = value
    return all_kv

def add_tag(conn: BlitzGateway, object_type: str, object_id: int, tag_text: str) -> int:
    tag_ann = omero.gateway.TagAnnotationWrapper(conn)
    tag_ann.setValue(tag_text)
    tag_ann.save()
    obj = conn.getObject(object_type, object_id)
    if not obj:
        raise ValueError(f"{object_type} {object_id} not found")
    obj.linkAnnotation(tag_ann)
    return tag_ann.getId()

def get_tags(conn: BlitzGateway, object_type: str, object_id: int) -> List[str]:
    obj = conn.getObject(object_type, object_id)
    if not obj:
        raise ValueError(f"{object_type} {object_id} not found")
    return [ann.getValue() for ann in obj.listAnnotations() if isinstance(ann, omero.gateway.TagAnnotationWrapper)]

def add_comment(conn: BlitzGateway, object_type: str, object_id: int, comment_text: str) -> int:
    comment_ann = omero.gateway.CommentAnnotationWrapper(conn)
    comment_ann.setValue(comment_text)
    comment_ann.save()
    obj = conn.getObject(object_type, object_id)
    if not obj:
        raise ValueError(f"{object_type} {object_id} not found")
    obj.linkAnnotation(comment_ann)
    return comment_ann.getId()

def get_comments(conn: BlitzGateway, object_type: str, object_id: int) -> List[str]:
    obj = conn.getObject(object_type, object_id)
    if not obj:
        raise ValueError(f"{object_type} {object_id} not found")
    return [ann.getValue() for ann in obj.listAnnotations() if isinstance(ann, omero.gateway.CommentAnnotationWrapper)]

## Examples

### Connect

In [9]:
load_dotenv("defualt.env")
load_dotenv(".env", override=True)
conn = connect_to_omero(os.getenv('OMERO_HOST'), os.getenv('OMERO_USERNAME'), os.getenv('OMERO_PASSWORD'))

2025-10-20 17:07:23,417 DEBUG [                           omero.gateway] (MainThread) localhost
2025-10-20 17:07:23,417 DEBUG [                           omero.gateway] (MainThread) 4064
2025-10-20 17:07:23,418 DEBUG [                           omero.gateway] (MainThread) []
2025-10-20 17:07:23,421 DEBUG [                           omero.gateway] (MainThread) Connect attempt, sUuid=None, group=None, self.sUuid=None
2025-10-20 17:07:23,421 DEBUG [                           omero.gateway] (MainThread) Creating Session...
2025-10-20 17:07:24,002 DEBUG [                     omero.gateway.utils] (MainThread) Setting 'omero.client.uuid' to '360856f4-9959-4b0d-b74c-6c6c1036474f'
2025-10-20 17:07:24,004 DEBUG [                     omero.gateway.utils] (MainThread) Setting 'omero.event' to 'Internal'
2025-10-20 17:07:24,004 DEBUG [                     omero.gateway.utils] (MainThread) Setting 'omero.session.uuid' to '36b0e5e9-618c-4550-ac09-730543b57e8c'
2025-10-20 17:07:24,005 DEBUG [         

### Upload

In [ ]:
project_id = create_project(conn, "My Project")
dataset_id = create_dataset(conn, "My Dataset", project_id=project_id)
image_id = upload_image(conn, "image.png", dataset_id)

2025-10-20 17:22:03,796 WARNI [                           omero.gateway] (MainThread) ConnectionLostException on <class 'omero.gateway.OmeroGatewaySafeCallWrapper'> to <360856f4-9959-4b0d-b74c-6c6c1036474fomero.api.IUpdate> saveAndReturnObject((object #0 (::omero::model::Project)
{
    _id = <nil>
    _details = object #1 (::omero::model::Details)
    {
        _owner = <nil>
        _group = <nil>
        _creationEvent = <nil>
        _updateEvent = <nil>
        _permissions = <nil>
        _externalInfo = <nil>
        _call = {}
        _event = <nil>
    }
    _loaded = True
    _version = <nil>
    _datasetLinksSeq = 
    {
    }
    _datasetLinksLoaded = True
    _datasetLinksCountPerOwner = {}
    _annotationLinksSeq = 
    {
    }
    _annotationLinksLoaded = True
    _annotationLinksCountPerOwner = {}
    _name = object #2 (::omero::RString)
    {
        _val = My Project
    }
    _description = <nil>
},), {})
Traceback (most recent call last):
  File "/opt/homebrew/Caskro

ConnectionLostException: Ice.ConnectionLostException:
recv() returned zero

In [13]:
print(dataset_id)

4


### Add Metadata

In [ ]:
add_key_value_pairs(conn, "Image", image_id, {"experiment": "test", "temp": "37C"})
add_tag(conn, "Image", image_id, "control")
add_comment(conn, "Image", image_id, "High quality image")

### Get Metadata

In [ ]:
kv_pairs = get_key_value_pairs(conn, "Image", image_id)
tags = get_tags(conn, "Image", image_id)
metadata = get_image_metadata(conn, image_id)

### Download

In [ ]:
download_image(conn, image_id, "/path/to/save/image.tif")
downloaded_files = download_dataset(conn, dataset_id, "/path/to/save/dir/")

### Dataset Info

In [ ]:
dataset_info = get_dataset_info(conn, dataset_id)

### Disconnect

In [ ]:
disconnect_from_omero(conn)